# Study Notes: Temporal Knowledge Graphs (TKGs)

## 1. Motivation and Core Concepts

Standard Knowledge Graphs (KGs) represent facts as eternal truths (e.g., `(Barack Obama, born_in, Honolulu)`). However, **change is the only constant**. Relationships often only hold true for specific periods (e.g., `(Barack Obama, president_of, USA)` from 2009 to 2017).

* **Temporal Tuple (Quintuple):** A standard KG triple is extended to include a time interval.
* Format: `(Subject, Relation, Object, Time_Start, Time_End)`
* **Time Granularity:** Depends on the application (e.g., milliseconds for Olympic records, millennia for the rise/fall of civilizations).
* **Time Variations:** Can be a single point (degenerate interval where start = end), a continuous interval, or a recurring instance (e.g., Olympic games every 4 years).



## 2. TKG Inference Tasks

When transitioning from static to temporal KGs, reasoning tasks become more complex:

1. **Temporal Link Prediction:** Predicting a missing subject or object given the time interval.
* *Positive Example:* `(?, president_of, USA, 2009, 2017)` $\rightarrow$ Barack Obama
* *Negative Example:* Providing "India" as the object during that interval is a negative fact.


2. **Time Prediction (New Task):** Predicting the specific time interval during which a given subject-relation-object triple holds true.
* *Example:* `(Barack Obama, president_of, USA, ?, ?)` $\rightarrow$ 2009–2017.



## 3. Evolution of TKG Completion Models

The core challenge in TKG completion is adapting the scoring function $F$ to accept start and end times. Does time modify the entity embeddings, or does it only interact at the scoring function level?

### A. HyTE (Hyperplane-based Temporal Embedding)

An early attempt to prevent anti-causal predictions (like a person getting married before they are born).

* **Mechanism:** Models every instant of time $\tau$ as a specific **hyperplane**. Entity and relation embeddings are projected onto this time-specific hyperplane.
* **Scoring:** Once projected onto the $\tau$ hyperplane, it relies on the **TransE** baseline: $E_{H,\tau} + R_{\tau} \approx E_{T,\tau}$ (Head + Relation $\approx$ Tail).
* **Drawback:** Because it is built on TransE, it inherits TransE's weaknesses, notably its inability to model many-to-many relationships effectively.

### B. T-ComplEx (Temporal ComplEx)

A stronger baseline leveraging **ComplEx** (which handles asymmetric and many-to-many relations well using complex vector spaces).

* **Standard ComplEx:** Uses a 3-way inner product between the Subject ($S$), Relation ($R$), and conjugate of the Object ($\bar{O}$). Score = $Re(\langle S, R, \bar{O} \rangle)$.
* **T-ComplEx Mechanism:** Extends this to a **4-way dot product** by embedding time ($T$) as a complex vector.
* **Drawback:** Time modulates exactly *one* of the inputs (S, R, or O) via element-wise multiplication. This incorrectly assumes *all* relationships are time-sensitive, which is not true (some facts are eternal).

### C. TNT-ComplEx

Designed to separate time-varying relations from timeless ones.

* **Mechanism:** Mixes two representations for a relationship:
* $R_T$: Time-sensitive representation.
* $R$: Time-independent (timeless/eternal) representation.


* **Formula Logic:** It combines $R_T$ with $T$ (element-wise), adds the timeless aspect $R$, and then takes the standard 3-way inner product with $S$ and $O$.
* **Advantage:** During training, if a relation is eternal, $R$ takes the representation burden and $R_T$ dwindles to a low-norm vector. If highly time-dependent, $R_T$ becomes highly informative.

### D. Timeplex (and Factor Graphs)

Enhances time-sensitivity by factoring in multiple contextual constraints, akin to a graphical model. It asks four independent questions to build confidence in a fact:

1. **S-O-R:** Can S and O be related by R (standard ComplEx)?
2. **S-R-T:** Can S participate in R at time T? *(e.g., Can a 10-year-old be a president?)*
3. **O-R-T:** Can O participate in R at time T? *(e.g., Did Google exist in 1980?)*
4. **S-O-T:** Can S and O interact at time T? *(e.g., Could Mozart and Elvis meet in 2005?)*

## 4. Modeling Temporal Constraints & Recurrence

Models strictly relying on scoring functions $F$ often lack common sense. Advanced TKG models inject auxiliary distributional knowledge:

* **Recurrence:** Capturing periodic events (e.g., US Elections every 4 years). The model detects a "blip in likelihood" in multiples of four.
* **Time Gaps (Distributions):** Modeling the gap between life events (e.g., Birth $\rightarrow$ College Graduation).
* *Mechanism:* Extract year differences across the KG (e.g., Biden graduated at 23, Kalam at 23).
* *Implementation:* Model this gap using a **Normal Distribution** (mean $\mu$, standard deviation $\sigma$).
* *Result:* When predicting a birth year based on a known graduation year, the model heavily penalizes guesses that fall outside the normal human life-cycle distribution, completely eliminating absurd predictions (like predicting marriage before birth).



## 5. Model Evaluation Metrics

* **Vector Distance of Time:** A well-trained model will naturally learn that the L2 distance between the complex embeddings of two years (e.g., 1960 and 2000) increases as the actual chronological gap increases.
* **aeIOU (Area Enhanced Intersection over Union):** Standard Mean Reciprocal Rank (MRR) is insufficient for time intervals. `aeIOU` is used to measure the intersection between the system-predicted time interval and the gold (true) time interval, penalizing the score based on the degree of mismatch.

---

## Practice Questions

**Q1: Why is TransE considered a weak baseline for the HyTE model in temporal knowledge graphs?**
A) It cannot handle multi-modal data.
B) It cannot effectively model many-to-many relationships.
C) It requires massive amounts of time intervals.
D) It relies on normal distributions for inference.
*Answer: B. (HyTE relies on TransE inside the time hyperplane, meaning it inherits TransE's inability to gracefully handle many-to-many relationships).*

**Q2: In the context of TNT-ComplEx, what happens to the representation vectors if a relationship is strictly "eternal" (time-independent)?**
A) $R_T$ carries the majority of the representation burden.
B) The time vector $T$ is dropped from the dataset.
C) The time-independent vector $R$ carries the representation burden, while $R_T$ dwindles to a low-norm/meaningless vector.
D) Both $R_T$ and $R$ become equivalent.
*Answer: C. (The model learns to ignore the time-varying component $R_T$ for timeless facts).*

**Q3: Which of the following is NOT one of the four contextual factors evaluated by Timeplex?**
A) Can S and O interact at time T? (S-O-T)
B) Can S participate in R at time T? (S-R-T)
C) Can S and O interact via R independent of time? (S-O-R)
D) Does T occur after the subject is born? (T > S_birth)
*Answer: D. (While temporal constraints handle logic like birth limits, the core structural factors of the Timeplex molecule are S-O-R, S-R-T, O-R-T, and S-O-T).*

---

## Revision Summary

* **TKGs** add time intervals to facts, introducing new tasks like **Time Prediction**.
* **HyTE** projects entities onto a time-specific hyperplane but relies on weak TransE logic.
* **T-ComplEx** uses a 4-way dot product but wrongly forces all facts to be time-sensitive.
* **TNT-ComplEx** solves this by splitting relations into time-varying ($R_T$) and timeless ($R$) components.
* **Timeplex** scores facts by analyzing multiple contextual subsets (e.g., S-R-T, O-R-T).
* **Temporal Constraints** (using Gaussian distributions for life-event gaps) and **Recurrence** signals are required to prevent absurd, anti-causal predictions.
* Use **aeIOU** (Area Enhanced Intersection over Union) to evaluate interval predictions, not standard MRR.